# Phase 8 — Evaluation
# Faithfulness vs Hallucination (Most Important Interview Topic ⭐⭐⭐⭐⭐)

This is one of the **most frequently asked interview questions** because both metrics are fundamental to evaluating LLMs and RAG systems.

Interviewers often ask:

- What is Faithfulness?
- What is Hallucination?
- Difference between Faithfulness and Hallucination?
- How do you reduce hallucinations?
- How do you evaluate them?

Although these concepts are related, they are **not identical**.

---

# 1. Faithfulness

## Definition

Faithfulness measures **whether the generated answer is completely supported by the retrieved context**.

In other words,

> **Did the LLM stay faithful to the provided documents?**

---

## Interview Definition

> Faithfulness measures whether every claim made by the LLM can be verified from the retrieved context. It evaluates whether the model remained grounded in the retrieved information.

---

# Example 1 (Faithful)

Retrieved Context

```text
Employees receive 20 annual leave days.

Unused leave can be carried forward for 5 days.
```

Question

```text
How many leave days are provided?
```

Answer

```text
Employees receive 20 annual leave days.
```

Faithfulness

```text
High (1.0)
```

Everything in the answer exists in the retrieved context.

---

# Example 2 (Not Faithful)

Retrieved Context

```text
Employees receive 20 annual leave days.
```

Answer

```text
Employees receive 25 annual leave days.
```

Faithfulness

```text
Low
```

The answer contains information not present in the context.

---

# Architecture

```text
User

↓

Retriever

↓

Context

↓

LLM

↓

Answer

↓

Faithfulness Check

↓

Supported?

↓

Yes / No
```

---

# Enterprise Example

Healthcare

Context

```text
Metformin should not be used in severe kidney disease.
```

Answer

```text
Metformin is safe for severe kidney disease.
```

Faithfulness

Low

---

# 2. Hallucination

## Definition

Hallucination occurs when an LLM **generates information that is false, unsupported, or completely fabricated**.

Hallucination is a **behavior of the model**.

---

## Interview Definition

> Hallucination occurs when an LLM generates facts, numbers, or statements that are not supported by the retrieved context or factual knowledge.

---

# Example

Context

```text
Employees receive 20 annual leave days.
```

Answer

```text
Employees receive 25 annual leave days.
```

The model invented

```text
25
```

Hallucination

High

---

Healthcare Example

Context

```text
No mention of Aspirin dosage.
```

Answer

```text
Recommended dosage is 500 mg twice daily.
```

The dosage was fabricated.

Hallucination

High

---

# Architecture

```text
User

↓

Retriever

↓

LLM

↓

Answer

↓

Fact Check

↓

Hallucination Score
```

---

# Relationship

Hallucination is the **problem**.

Faithfulness is the **metric used to detect that problem**.

Think of it like:

```text
Hallucination

↓

Wrong Information

↓

Faithfulness Metric

↓

Detects Wrong Information
```

---

# 3. Side-by-Side Example

Retrieved Context

```text
Employees receive 20 annual leave days.

Carry forward is 5 days.
```

Question

```text
How many leave days are provided?
```

Answer

```text
Employees receive 20 annual leave days.
```

| Metric | Result |
|---------|--------|
| Faithfulness | High |
| Hallucination | Low |

---

Second Example

Retrieved Context

```text
Employees receive 20 annual leave days.
```

Answer

```text
Employees receive 30 annual leave days.
```

| Metric | Result |
|---------|--------|
| Faithfulness | Low |
| Hallucination | High |

---

# 4. Are They Opposite?

Almost, but not exactly.

Usually

```text
High Faithfulness

↓

Low Hallucination
```

and

```text
Low Faithfulness

↓

High Hallucination
```

However, an answer can be:

- Faithful but incomplete.
- Hallucination-free but still not answer the user's question.

Example

Question

```text
How many leave days are provided?
```

Context

```text
Employees receive 20 annual leave days.
```

Answer

```text
The company has a leave policy.
```

Hallucination

Low (nothing invented)

Faithfulness

Reasonably high (supported by context)

But

Answer Relevancy

Low

This is why multiple evaluation metrics are needed.

---

# 5. Enterprise Architecture

```text
                  User

                    │

                    ▼

             FastAPI API

                    │

                    ▼

               LangGraph

                    │

                    ▼

             Hybrid Search

                    │

                    ▼

         OpenSearch / AI Search

                    │

                    ▼

          AWS Bedrock / Azure OpenAI

                    │

                    ▼

                 Response

                    │

          ┌─────────┴─────────┐

          ▼                   ▼

   Faithfulness         Hallucination

      Evaluation          Detection
```

---

# AWS + Azure Components

| Layer | AWS | Azure |
|--------|------|--------|
| LLM | Bedrock | Azure OpenAI |
| Retriever | OpenSearch | Azure AI Search |
| Evaluation | RAGAS / DeepEval | RAGAS / DeepEval |
| Observability | LangSmith | LangSmith |

---

# 6. LangChain Example

Suppose

Context

```text
Employees receive 20 annual leave days.
```

LLM

↓

Answer

```text
Employees receive 25 annual leave days.
```

Evaluation

```python
Faithfulness = 0.32

Hallucination = 0.91
```

Meaning

Very poor answer.

---

# 7. DeepEval Example

```python
from deepeval import evaluate
from deepeval.metrics import (
    FaithfulnessMetric,
    HallucinationMetric
)
from deepeval.test_case import LLMTestCase

# ==========================================================
# STEP 1 : Create Test Case
# ==========================================================

test_case = LLMTestCase(
    input="How many annual leave days are provided?",
    actual_output="Employees receive 25 annual leave days.",
    retrieval_context=[
        "Employees receive 20 annual leave days."
    ]
)

# ==========================================================
# STEP 2 : Create Evaluation Metrics
# ==========================================================

faithfulness = FaithfulnessMetric()

hallucination = HallucinationMetric()

# ==========================================================
# STEP 3 : Evaluate
# ==========================================================

evaluate(
    test_cases=[test_case],
    metrics=[
        faithfulness,
        hallucination
    ]
)
```

Expected outcome

```text
Faithfulness  : FAIL

Hallucination : FAIL
```

---

# 8. How to Reduce Hallucination?

### 1. Better Retrieval

```text
Semantic Chunking

↓

Hybrid Search

↓

Parent Child Retrieval
```

---

### 2. Better Prompts

Example

```text
Answer ONLY using the retrieved context.

If the answer is unavailable,

say

"I don't know."
```

---

### 3. Better Chunking

Use

- Semantic Chunking
- Parent Child Retrieval

---

### 4. Better Embeddings

Use

- Titan Embeddings
- text-embedding-3-large
- BGE
- E5

---

### 5. Reranking

Keep only relevant chunks.

---

### 6. Context Compression

Remove irrelevant information.

---

### 7. Guardrails

Reject unsupported answers.

---

### 8. Human Review

High-risk domains

- Healthcare
- Banking
- Legal

---

# 9. Best Practices

✅ Use Hybrid Search.

✅ Use Query Rewriting.

✅ Use Reranking.

✅ Add Context Compression.

✅ Use Guardrails.

✅ Evaluate continuously.

---

# 10. Common Mistakes

❌ Sending irrelevant documents.

❌ Using poor chunk sizes.

❌ No retrieval evaluation.

❌ Trusting every LLM response.

---

# 11. Comparison Table

| Feature | Faithfulness | Hallucination |
|----------|--------------|---------------|
| Type | Evaluation Metric | Model Behavior |
| Measures | Answer grounded in context | Fabricated information |
| Good Score | High | Low |
| Evaluated By | RAGAS, DeepEval | DeepEval, RAGAS (via faithfulness-related checks) |
| Enterprise Goal | Maximize | Minimize |

---

# 12. Common Interview Questions

### Q1. What is Faithfulness?

It measures whether the generated answer is fully supported by the retrieved context.

---

### Q2. What is Hallucination?

Hallucination occurs when the model generates information that is not supported by the context or facts.

---

### Q3. Can an answer be hallucination-free but still poor?

Yes.

Example

Question

```text
How many leave days?
```

Answer

```text
The company has a leave policy.
```

Nothing is fabricated, so hallucination is low.

But it doesn't answer the question, so answer relevancy is poor.

---

### Q4. Which tools evaluate these metrics?

- RAGAS
- DeepEval
- LangSmith (through integrations and custom evaluators)

---

### Q5. How do you reduce hallucinations in production?

- Hybrid Search
- Semantic Chunking
- Parent-Child Retrieval
- Query Rewriting
- Reranking
- Context Compression
- Strong prompting
- Guardrails
- Continuous evaluation with RAGAS and DeepEval

---

# 13. EPAM Senior Answer (3–4 Minutes)

> "Faithfulness and hallucination are closely related but represent different concepts. Faithfulness is an evaluation metric that measures whether the generated answer is completely supported by the retrieved context. Hallucination is the behavior of an LLM when it generates unsupported or fabricated information. For example, if the retrieved HR policy states that employees receive 20 annual leave days but the model answers 25 days, the response has low faithfulness and high hallucination. In production RAG systems, I reduce hallucinations by improving retrieval quality through Semantic Chunking, Parent-Child Retrieval, Hybrid Search, Query Rewriting, reranking, and Context Compression. I also use prompts that instruct the model to answer only from the provided context and respond with 'I don't know' when the information is unavailable. Finally, I continuously evaluate the system using RAGAS and DeepEval, while LangSmith provides execution traces to help identify where hallucinations originate."